# Interactive FSDP Training Job

Select your normal **Python 3** kernel and set **Processes: 2** before running. This notebook builds a deliberately tiny FSDP Qwen model, applies PyTorch composable FSDP, takes one optimizer step, pauses to inspect rank-local state, and then continues with the same live model and optimizer.

The demo showcases how a distributed model can retain the familiar notebook workflow: run it, inspect it, and continue working with it just as you would a non-distributed model. Qwen3-0.6B keeps the example approachable, but try replacing it with the largest model that fits on your machine.

> This notebook assumes your environment contains the following dependencies (apart from the jupyter-distributed extension):
> - torch
> - transformers
> - datasets
> - tqdm
> - ipywidgets

In [ ]:
import os

import torch
import torch.distributed as dist

rank = int(os.environ["RANK"])
local_rank = int(os.environ["LOCAL_RANK"])
world_size = int(os.environ["WORLD_SIZE"])
assert world_size >= 2, "Set Processes to at least 2 and restart the kernel group"
assert torch.cuda.device_count() >= world_size, "This demo needs one CUDA GPU per process"
torch.cuda.set_device(local_rank)
device = torch.device("cuda", local_rank)
if not dist.is_initialized():
    dist.init_process_group("nccl")
print(f"rank={rank}/{world_size} device={device}")

Load the tokenizer and the smallest Qwen3 causal language model with Transformers. Each process initially loads the same pretrained model before FSDP shards its parameters.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)
model.config.use_cache = False
model.to(device)
print(f"rank {rank}: loaded {model_id} with {model.num_parameters():,} parameters")

Create a one-dimensional device mesh and apply composable FSDP from the inside out. The mesh describes the participating devices and names the data-parallel dimension; `fully_shard` supplies the sharding behavior. Sharding each transformer block before the root model creates practical per-block communication groups instead of treating the entire model as one FSDP unit.

In [ ]:
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.fsdp import fully_shard

mesh = init_device_mesh("cuda", (world_size,), mesh_dim_names=("dp",))
for block in model.model.layers:
    fully_shard(block, mesh=mesh)
fully_shard(model, mesh=mesh)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
print(f"rank {rank}: mesh={mesh} model_type={type(model).__name__}")

Load the LIMA training split.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("llamafactory/lima", split="train")
dataset

Convert each ShareGPT-style conversation to Qwen's chat format and cap the sequence length for this interactive example.

In [ ]:
role_map = {"human": "user", "gpt": "assistant"}


def tokenize_conversation(example):
    messages = [
        {"role": role_map[turn["from"]], "content": turn["value"]}
        for turn in example["conversations"]
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        truncation=True,
        max_length=512,
        return_dict=False,
    )
    return {"input_ids": input_ids}


tokenized_dataset = dataset.map(
    tokenize_conversation,
    remove_columns=dataset.column_names,
)
len(tokenized_dataset)

Give process `rank` the examples whose indices satisfy `index % world_size == rank`. The incomplete final group is dropped so every process performs the same number of FSDP steps.

In [ ]:
from torch.utils.data import DataLoader, Subset
from transformers import DataCollatorForLanguageModeling

usable_examples = len(tokenized_dataset) - len(tokenized_dataset) % world_size
rank_indices = range(rank, usable_examples, world_size)
rank_dataset = Subset(tokenized_dataset, rank_indices)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
train_loader = DataLoader(rank_dataset, batch_size=1, collate_fn=collator)
train_batches = iter(train_loader)
print(f"rank {rank}: {len(rank_dataset)} examples")

Take one optimizer step and retain the model, optimizer, data iterator, and loss for later cells.

In [ ]:
def train_step(batch):
    batch = {name: value.to(device) for name, value in batch.items()}
    optimizer.zero_grad(set_to_none=True)
    loss = model(**batch).loss
    loss.backward()
    optimizer.step()
    return float(loss.detach())


first_loss = train_step(next(train_batches))
print(f"rank {rank}: first_loss={first_loss:.4f}")

Pause here and inspect the still-live distributed job. Rank tabs show each process's local parameter shard while `model`, `optimizer`, `train_batches`, and `first_loss` remain available from earlier cells.

In [ ]:
name, parameter = next(iter(model.named_parameters()))
local = parameter.to_local() if hasattr(parameter, "to_local") else parameter
print(
    {
        "rank": rank,
        "parameter": name,
        "global_shape": tuple(parameter.shape),
        "local_shape": tuple(local.shape),
        "placements": tuple(getattr(parameter, "placements", ())),
        "local_norm": float(local.float().norm()),
        "first_loss": first_loss,
    }
)

Optionally compile the live FSDP model before continuing. Skip the next cell for an eager baseline, or run it and compare the training rate in the final progress bar. PyTorch compiles lazily, so the first subsequent step includes compilation overhead while later steps show steady-state performance.

In [ ]:
model.compile()

Continue through the rest of each process's dataset partition without rebuilding any state. Each rank owns an independent live progress widget.

In [ ]:
from tqdm.notebook import tqdm

later_losses = []
for batch in tqdm(
    train_batches,
    total=len(train_loader) - 1,
    desc=f"rank {rank}",
    unit="step",
):
    later_losses.append(train_step(batch))
print(
    {
        "rank": rank,
        "steps": 1 + len(later_losses),
        "first_loss": first_loss,
        "last_loss": later_losses[-1],
    }
)